<a href="https://colab.research.google.com/github/tarunsac/Gen-AI/blob/main/DL_using_PyTorch_Prediction_Water_Quality.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In this project, we provide a water quality dataset from the Central Pollution Control Board (CPCB). You will use this dataset to predict water quality.

Access to clean water is a fundamental necessity, yet water quality can vary significantly based on environmental, geographical, and human factors. The dataset includes water quality monitoring data from across India, with chemical and physical parameters measured at various locations over different years (2019, 2020, 2021, 2022).

**The dataset contains the following columns**:
* Well_ID, State, District, Block, Village: Geographical identifiers of water sampling locations.
* Latitude, Longitude: Spatial coordinates for precise mapping.
* Year: The year in which the sample was recorded.
* Water Quality Indicators: pH, Electrical Conductivity (EC), Carbonates (CO3), Bicarbonates (HCO3), Chlorides (Cl), Sulfates (SO4), Nitrates (NO3), Total Hardness (TH), Calcium (Ca), Magnesium (Mg), Sodium (Na), Potassium (K), Fluoride (F), Total Dissolved Solids (TDS).

**Target Variables**:
* Water Quality Index (WQI): A numerical representation of overall water quality derived from chemical and physical parameters.
* Water Quality Classification: A categorical label indicating the quality of water (e.g., Good, Poor, Unsuitable for Drinking).
* In this project, your task is to build Deep Learning Neural Networks to predict the following:
  * Water Quality Index (WQI)
  * Water Quality Classification

Use the dataset from the attachment provided and build the models using that. Once the models are built, calculate metrics such as the R² score for regression and accuracy/F1 score for classification

# Task
Analyze the `water_quality.csv` dataset, preprocess the data, and then build and evaluate two deep learning neural networks: one for predicting the Water Quality Index (WQI) (regression) and another for predicting Water Quality Classification (classification). Finally, summarize the performance of both models using R² score for regression and accuracy/F1 score for classification.

## Load and Initial Data Exploration

### Subtask:
Load the 'water_quality.csv' dataset into a pandas DataFrame. Perform initial data exploration to understand its structure, identify missing values, and analyze data types.


**Reasoning**:
I need to load the 'water_quality.csv' dataset, display its initial rows, check data types, identify missing values, and generate descriptive statistics as requested in the instructions.



In [1]:
import pandas as pd

# Load the dataset
df = pd.read_csv('/content/water_quality.csv')
print("Dataset loaded successfully.")

# Display the first 5 rows
print("\nFirst 5 rows of the DataFrame:")
print(df.head())

# Check data types and non-null values
print("\nDataFrame Info:")
df.info()

# Identify and count missing values
print("\nMissing values count per column:")
print(df.isnull().sum())

# Generate descriptive statistics for numerical columns
print("\nDescriptive statistics for numerical columns:")
print(df.describe())

Dataset loaded successfully.

First 5 rows of the DataFrame:
            Well_ID             State    District      Block    Village  \
0  W232200071580001           Gujarat   Ahmedabad     Mandal      Dalod   
1  W254029084355301  Himachal Pradesh       Solan  Nallagarh   JAGATPUR   
2  W193530074180001       Maharashtra  Ahmednagar  SANGAMNER  Kokangaon   
3  W231620072033001           Gujarat   Ahmedabad   Viramgam      Endla   
4  W251908084361501  Himachal Pradesh       Solan  Nallagarh     BARUNA   

   Latitude  Longitude  Year    pH       EC  ...    NO3      TH     Ca     Mg  \
0       NaN        NaN  2020  8.20  16640.0  ...  26.00  1451.0  152.0  260.0   
1   31.1594  76.678500  2019  8.44    299.0  ...   2.70    84.0   17.0   10.0   
2   74.3000  19.591667  2022  7.90   1315.0  ...  18.20   465.0   80.2   64.4   
3       NaN        NaN  2020  7.40    715.0  ...   0.23   280.0   56.0   34.0   
4   31.1540  76.638400  2019  8.40    270.0  ...  10.00   116.0   10.0   18.0   

 

## Data Preprocessing and Feature Engineering

### Subtask:
Handle missing values, encode categorical features (State, District, Block, Village, Year), and scale numerical features. Create target variables for WQI (regression) and Water Quality Classification. Convert the processed data into PyTorch tensors and split into training and testing sets.


**Reasoning**:
First, I'll drop the 'Well_ID' column, fill missing string values in 'Block' and 'Village', and fill missing numerical values in 'Latitude' and 'Longitude' with their medians. Then, I'll address negative 'F' values and separate features from target variables as specified in the instructions. This prepares the data for encoding and scaling.



In [2]:
from sklearn.preprocessing import LabelEncoder

# 1. Drop the 'Well_ID' column
df_processed = df.drop('Well_ID', axis=1)
print("Dropped 'Well_ID' column.")

# 2. Fill missing string values in 'Block' and 'Village' with 'Unknown'
df_processed['Block'] = df_processed['Block'].fillna('Unknown')
df_processed['Village'] = df_processed['Village'].fillna('Unknown')
print("Filled missing values in 'Block' and 'Village' with 'Unknown'.")

# 3. Fill missing numerical values in 'Latitude' and 'Longitude' with the median
df_processed['Latitude'] = df_processed['Latitude'].fillna(df_processed['Latitude'].median())
df_processed['Longitude'] = df_processed['Longitude'].fillna(df_processed['Longitude'].median())
print("Filled missing values in 'Latitude' and 'Longitude' with their medians.")

# 4. Address anomalous negative values in the 'F' (Fluoride) column
df_processed['F'] = df_processed['F'].apply(lambda x: max(x, 0))
print("Replaced negative values in 'F' column with 0.")

# 5. Select the features for the models. Exclude 'Water Quality Classification' and 'WQI' from the feature set.
feature_columns = df_processed.drop(columns=['WQI', 'Water Quality Classification']).columns
X = df_processed[feature_columns]
print(f"Selected {len(feature_columns)} feature columns.")

# 6. Create the regression target variable, y_wqi, using the 'WQI' column.
y_wqi = df_processed['WQI']
print("Created regression target variable 'y_wqi'.")

# 7. Create the classification target variable, y_classification, using the 'Water Quality Classification' column.
# Convert these categorical string labels into numerical labels using LabelEncoder.
le = LabelEncoder()
y_classification = le.fit_transform(df_processed['Water Quality Classification'])
print("Created classification target variable 'y_classification' and encoded it.")
print(f"Original classification labels: {df_processed['Water Quality Classification'].unique()}")
print(f"Encoded classification labels: {le.transform(df_processed['Water Quality Classification'].unique())}")

Dropped 'Well_ID' column.
Filled missing values in 'Block' and 'Village' with 'Unknown'.
Filled missing values in 'Latitude' and 'Longitude' with their medians.
Replaced negative values in 'F' column with 0.
Selected 21 feature columns.
Created regression target variable 'y_wqi'.
Created classification target variable 'y_classification' and encoded it.
Original classification labels: ['Unsuitable for Drinking' 'Good' 'Very Poor yet Drinkable' 'Poor'
 'Excellent']
Encoded classification labels: [3 1 4 2 0]


**Reasoning**:
Now that initial data cleaning and target variable creation are complete, I will proceed with encoding categorical features using one-hot encoding and scaling numerical features using StandardScaler. This prepares the data for model training by converting all features into a suitable numerical format and standardizing their ranges.



In [3]:
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer

# Identify categorical and numerical columns from X
categorical_features = ['State', 'District', 'Block', 'Village', 'Year']
numerical_features = X.select_dtypes(include=['float64', 'int64']).columns.tolist()

# Remove 'Year' from numerical features as it's being treated as categorical for one-hot encoding
if 'Year' in numerical_features:
    numerical_features.remove('Year')

print(f"Categorical features identified: {categorical_features}")
print(f"Numerical features identified: {numerical_features}")

# Create a column transformer for one-hot encoding categorical features and scaling numerical features
# Use handle_unknown='ignore' for OneHotEncoder to avoid errors during transformation if an unseen category is encountered
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numerical_features),
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features)
    ],
    remainder='passthrough' # Keep any other columns not specified (shouldn't be any in this case)
)

# Apply the preprocessing steps to the features (X)
X_processed = preprocessor.fit_transform(X)

print("Categorical features one-hot encoded and numerical features scaled.")
print(f"Shape of processed features X_processed: {X_processed.shape}")

Categorical features identified: ['State', 'District', 'Block', 'Village', 'Year']
Numerical features identified: ['Latitude', 'Longitude', 'pH', 'EC', 'CO3', 'HCO3', 'Cl', 'SO4', 'NO3', 'TH', 'Ca', 'Mg', 'Na', 'K', 'F', 'TDS']
Categorical features one-hot encoded and numerical features scaled.
Shape of processed features X_processed: (19029, 16901)


**Reasoning**:
With the features preprocessed and target variables ready, I will now split the data into training and testing sets for both regression and classification tasks. Following that, I will convert these sets into PyTorch tensors, ensuring the correct data types for neural network input.



In [4]:
from sklearn.model_selection import train_test_split
import torch

# Split the processed features and regression target into training and testing sets
X_train_wqi, X_test_wqi, y_train_wqi, y_test_wqi = train_test_split(
    X_processed, y_wqi, test_size=0.2, random_state=42
)
print("Data split for WQI regression completed.")

# Split the processed features and classification target into training and testing sets
X_train_cls, X_test_cls, y_train_cls, y_test_cls = train_test_split(
    X_processed, y_classification, test_size=0.2, random_state=42
)
print("Data split for Water Quality Classification completed.")

# Convert NumPy arrays to PyTorch tensors
# Features for both tasks (WQI and Classification) are the same, so we can use one set of X tensors
X_train_tensor = torch.tensor(X_train_wqi.toarray() if hasattr(X_train_wqi, 'toarray') else X_train_wqi, dtype=torch.float32)
X_test_tensor = torch.tensor(X_test_wqi.toarray() if hasattr(X_test_wqi, 'toarray') else X_test_wqi, dtype=torch.float32)

# Target for WQI regression
y_train_wqi_tensor = torch.tensor(y_train_wqi.values if hasattr(y_train_wqi, 'values') else y_train_wqi, dtype=torch.float32).unsqueeze(1)
y_test_wqi_tensor = torch.tensor(y_test_wqi.values if hasattr(y_test_wqi, 'values') else y_test_wqi, dtype=torch.float32).unsqueeze(1)

# Target for Water Quality Classification
y_train_cls_tensor = torch.tensor(y_train_cls, dtype=torch.long)
y_test_cls_tensor = torch.tensor(y_test_cls, dtype=torch.long)

print("All data converted to PyTorch tensors.")

print(f"X_train_tensor shape: {X_train_tensor.shape}")
print(f"y_train_wqi_tensor shape: {y_train_wqi_tensor.shape}")
print(f"y_train_cls_tensor shape: {y_train_cls_tensor.shape}")
print(f"X_test_tensor shape: {X_test_tensor.shape}")
print(f"y_test_wqi_tensor shape: {y_test_wqi_tensor.shape}")
print(f"y_test_cls_tensor shape: {y_test_cls_tensor.shape}")

Data split for WQI regression completed.
Data split for Water Quality Classification completed.
All data converted to PyTorch tensors.
X_train_tensor shape: torch.Size([15223, 16901])
y_train_wqi_tensor shape: torch.Size([15223, 1])
y_train_cls_tensor shape: torch.Size([15223])
X_test_tensor shape: torch.Size([3806, 16901])
y_test_wqi_tensor shape: torch.Size([3806, 1])
y_test_cls_tensor shape: torch.Size([3806])


## Build and Train Regression Model (WQI)

### Subtask:
Define a Deep Learning Neural Network using PyTorch for predicting the Water Quality Index (WQI). Train the model, monitoring its performance during training.


**Reasoning**:
I need to define a PyTorch neural network for WQI regression, instantiate it, set up the loss function and optimizer, and then implement the training loop as per the instructions provided.



In [5]:
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader

# 2. Define a neural network class for regression
class RegressionModel(nn.Module):
    def __init__(self, input_size):
        super(RegressionModel, self).__init__()
        self.layer1 = nn.Linear(input_size, 256)
        self.relu1 = nn.ReLU()
        self.layer2 = nn.Linear(256, 128)
        self.relu2 = nn.ReLU()
        self.layer3 = nn.Linear(128, 64)
        self.relu3 = nn.ReLU()
        self.output_layer = nn.Linear(64, 1) # Single output for WQI

    def forward(self, x):
        x = self.relu1(self.layer1(x))
        x = self.relu2(self.layer2(x))
        x = self.relu3(self.layer3(x))
        x = self.output_layer(x)
        return x

# 5. Instantiate the RegressionModel
input_size_wqi = X_train_tensor.shape[1]
model_wqi = RegressionModel(input_size_wqi)
print(f"Regression model (WQI) instantiated with input size: {input_size_wqi}")

# 6. Define the loss function for regression
criterion_wqi = nn.MSELoss()
print("Loss function (MSELoss) defined for WQI regression.")

# 7. Define the optimizer (Adam) for WQI regression
optimizer_wqi = optim.Adam(model_wqi.parameters(), lr=0.001)
print("Optimizer (Adam) defined for WQI regression.")

# 8. Define the number of training epochs and a batch size
num_epochs_wqi = 50
batch_size_wqi = 64
print(f"Training configured for {num_epochs_wqi} epochs with batch size {batch_size_wqi}.")

# 9. Create a TensorDataset for WQI regression
train_dataset_wqi = TensorDataset(X_train_tensor, y_train_wqi_tensor)
print("TensorDataset created for WQI training data.")

# 10. Create a DataLoader for WQI regression
train_loader_wqi = DataLoader(dataset=train_dataset_wqi, batch_size=batch_size_wqi, shuffle=True)
print("DataLoader created for WQI training data.")

# 11. Implement the training loop for WQI regression
print("Starting WQI regression model training...")
for epoch in range(num_epochs_wqi):
    model_wqi.train() # Set model to training mode
    running_loss = 0.0
    for i, (inputs, labels) in enumerate(train_loader_wqi):
        # Zero the parameter gradients
        optimizer_wqi.zero_grad()

        # Forward pass
        outputs = model_wqi(inputs)
        loss = criterion_wqi(outputs, labels)

        # Backward and optimize
        loss.backward()
        optimizer_wqi.step()

        running_loss += loss.item() * inputs.size(0)

    epoch_loss = running_loss / len(train_dataset_wqi)
    print(f'Epoch [{epoch+1}/{num_epochs_wqi}], Loss: {epoch_loss:.4f}')
print("WQI regression model training complete.")

Regression model (WQI) instantiated with input size: 16901
Loss function (MSELoss) defined for WQI regression.
Optimizer (Adam) defined for WQI regression.
Training configured for 50 epochs with batch size 64.
TensorDataset created for WQI training data.
DataLoader created for WQI training data.
Starting WQI regression model training...
Epoch [1/50], Loss: 49205.0851
Epoch [2/50], Loss: 705.9520
Epoch [3/50], Loss: 289.7899
Epoch [4/50], Loss: 126.7826
Epoch [5/50], Loss: 76.3584
Epoch [6/50], Loss: 55.2664
Epoch [7/50], Loss: 54.6094
Epoch [8/50], Loss: 60.2346
Epoch [9/50], Loss: 54.7501
Epoch [10/50], Loss: 45.2140
Epoch [11/50], Loss: 44.5430
Epoch [12/50], Loss: 46.9544
Epoch [13/50], Loss: 44.5207
Epoch [14/50], Loss: 50.4575
Epoch [15/50], Loss: 55.9929
Epoch [16/50], Loss: 126.2275
Epoch [17/50], Loss: 90.6492
Epoch [18/50], Loss: 53.9251
Epoch [19/50], Loss: 50.8425
Epoch [20/50], Loss: 37.2259
Epoch [21/50], Loss: 31.2593
Epoch [22/50], Loss: 32.4566
Epoch [23/50], Loss: 40.6

## Evaluate Regression Model

### Subtask:
Evaluate the trained regression model on the test set and calculate the R² score.


**Reasoning**:
I need to evaluate the trained WQI regression model by making predictions on the test set, converting results to NumPy, and then calculating the R² score as specified in the instructions.



In [6]:
from sklearn.metrics import r2_score

# 1. Set the model to evaluation mode
model_wqi.eval()

# 2. Disable gradient calculations for inference
with torch.no_grad():
    # 3. Make predictions on the X_test_tensor
    y_pred_wqi_tensor = model_wqi(X_test_tensor)

# 4. Convert the predicted WQI values and true WQI values to NumPy arrays
y_pred_wqi_np = y_pred_wqi_tensor.numpy()
y_test_wqi_np = y_test_wqi_tensor.numpy()

# 5. Calculate the R² score
r2 = r2_score(y_test_wqi_np, y_pred_wqi_np)

# 6. Print the calculated R² score
print(f"R² score for WQI Regression Model: {r2:.4f}")

R² score for WQI Regression Model: 0.9994


## Build and Train Classification Model

### Subtask:
Define a Deep Learning Neural Network using PyTorch for predicting Water Quality Classification. Train the model, ensuring to handle potential class imbalance if present.


**Reasoning**:
I need to define the classification model, instantiate it, calculate class weights for CrossEntropyLoss to handle potential class imbalance, set up the optimizer, create the DataLoader, and implement the training loop as per the instructions.



In [7]:
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader

# 1. Define a neural network class for classification
class ClassificationModel(nn.Module):
    def __init__(self, input_size, num_classes):
        super(ClassificationModel, self).__init__()
        self.layer1 = nn.Linear(input_size, 256)
        self.relu1 = nn.ReLU()
        self.layer2 = nn.Linear(256, 128)
        self.relu2 = nn.ReLU()
        self.layer3 = nn.Linear(128, 64)
        self.relu3 = nn.ReLU()
        self.output_layer = nn.Linear(64, num_classes) # Output for number of classes

    def forward(self, x):
        x = self.relu1(self.layer1(x))
        x = self.relu2(self.layer2(x))
        x = self.relu3(self.layer3(x))
        x = self.output_layer(x)
        return x

# 2. Determine the number of unique classes
num_classes = y_train_cls_tensor.unique().size(0)
print(f"Number of unique classes: {num_classes}")

# 3. Instantiate the ClassificationModel
input_size_cls = X_train_tensor.shape[1]
model_cls = ClassificationModel(input_size_cls, num_classes)
print(f"Classification model instantiated with input size: {input_size_cls} and {num_classes} classes.")

# 4. Define the loss function (CrossEntropyLoss) and handle class imbalance
# Calculate class weights to handle potential class imbalance
class_counts = torch.bincount(y_train_cls_tensor)
class_weights = 1. / class_counts.float()
class_weights = class_weights / class_weights.sum() # Normalize weights to sum to 1
# Scale weights inversely proportional to class frequency
# A more common approach is to use the inverse of the class frequency as weights.
# class_weights = len(y_train_cls_tensor) / (len(class_counts) * class_counts.float())
# Let's try the simpler inverse frequency first, as CrossEntropyLoss expects non-normalized weights
class_weights_unnormalized = len(y_train_cls_tensor) / (class_counts.float())

criterion_cls = nn.CrossEntropyLoss(weight=class_weights_unnormalized)
print("Loss function (CrossEntropyLoss) defined for classification with class weights.")
print(f"Class counts: {class_counts.numpy()}, Class weights for CrossEntropyLoss: {class_weights_unnormalized.numpy()}")

# 5. Define the optimizer (Adam) for classification
optimizer_cls = optim.Adam(model_cls.parameters(), lr=0.001)
print("Optimizer (Adam) defined for classification.")

# 6. Define the number of training epochs and a batch size
num_epochs_cls = 50
batch_size_cls = 64
print(f"Training configured for {num_epochs_cls} epochs with batch size {batch_size_cls}.")

# 7. Create a TensorDataset for classification
train_dataset_cls = TensorDataset(X_train_tensor, y_train_cls_tensor)
print("TensorDataset created for classification training data.")

# 8. Create a DataLoader for classification
train_loader_cls = DataLoader(dataset=train_dataset_cls, batch_size=batch_size_cls, shuffle=True)
print("DataLoader created for classification training data.")

# 9-11. Implement the training loop for classification
print("Starting Water Quality Classification model training...")
for epoch in range(num_epochs_cls):
    model_cls.train() # Set model to training mode
    running_loss_cls = 0.0
    for i, (inputs, labels) in enumerate(train_loader_cls):
        # Zero the parameter gradients
        optimizer_cls.zero_grad()

        # Forward pass
        outputs = model_cls(inputs)
        loss = criterion_cls(outputs, labels)

        # Backward and optimize
        loss.backward()
        optimizer_cls.step()

        running_loss_cls += loss.item() * inputs.size(0)

    epoch_loss_cls = running_loss_cls / len(train_dataset_cls)
    print(f'Epoch [{epoch+1}/{num_epochs_cls}], Loss: {epoch_loss_cls:.4f}')
print("Water Quality Classification model training complete.")

Number of unique classes: 5
Classification model instantiated with input size: 16901 and 5 classes.
Loss function (CrossEntropyLoss) defined for classification with class weights.
Class counts: [ 627 1315 4251 5238 3792], Class weights for CrossEntropyLoss: [24.279106  11.576426   3.5810397  2.906262   4.014504 ]
Optimizer (Adam) defined for classification.
Training configured for 50 epochs with batch size 64.
TensorDataset created for classification training data.
DataLoader created for classification training data.
Starting Water Quality Classification model training...
Epoch [1/50], Loss: 0.5772
Epoch [2/50], Loss: 0.1743
Epoch [3/50], Loss: 0.0737
Epoch [4/50], Loss: 0.0547
Epoch [5/50], Loss: 0.0471
Epoch [6/50], Loss: 0.0474
Epoch [7/50], Loss: 0.0380
Epoch [8/50], Loss: 0.0312
Epoch [9/50], Loss: 0.0261
Epoch [10/50], Loss: 0.0222
Epoch [11/50], Loss: 0.0189
Epoch [12/50], Loss: 0.0238
Epoch [13/50], Loss: 0.0197
Epoch [14/50], Loss: 0.0232
Epoch [15/50], Loss: 0.0120
Epoch [16/

**Reasoning**:
I need to evaluate the trained Water Quality Classification model by making predictions on the test set, converting results to NumPy, and then calculating accuracy and F1 score, as specified in the instructions.



In [8]:
from sklearn.metrics import accuracy_score, f1_score

# 1. Set the classification model to evaluation mode
model_cls.eval()

# 2. Disable gradient calculations for inference
with torch.no_grad():
    # 3. Make predictions on the X_test_tensor
    outputs_cls = model_cls(X_test_tensor)

    # Get the predicted class by taking the argmax of the output logits
    _, y_pred_cls_tensor = torch.max(outputs_cls, 1)

# 4. Convert the predicted classification labels and true labels to NumPy arrays
y_pred_cls_np = y_pred_cls_tensor.numpy()
y_test_cls_np = y_test_cls_tensor.numpy()

# 5. Calculate Accuracy and F1-score
accuracy = accuracy_score(y_test_cls_np, y_pred_cls_np)
f1 = f1_score(y_test_cls_np, y_pred_cls_np, average='weighted') # Use 'weighted' for multi-class imbalance

# 6. Print the calculated metrics
print(f"Accuracy for Water Quality Classification Model: {accuracy:.4f}")
print(f"F1 Score (weighted) for Water Quality Classification Model: {f1:.4f}")

Accuracy for Water Quality Classification Model: 0.8815
F1 Score (weighted) for Water Quality Classification Model: 0.8837


## Final Task

### Subtask:
Summarize the performance of both the WQI regression model (R² score) and the Water Quality Classification model (accuracy, F1 score).


## Summary:

### Q&A
The WQI regression model achieved an R² score of 0.9994. The Water Quality Classification model achieved an accuracy of 0.8815 and a weighted F1 score of 0.8837.

### Data Analysis Key Findings
*   **Initial Data State**: The dataset contained 19029 entries and 24 columns, with missing values identified in `Well_ID` (3785), `Block` (1119), `Latitude` (389), `Longitude` (390), and `Village` (1). Anomalies included extreme values in `Latitude` and `Longitude`, and physically impossible negative values in the 'F' (Fluoride) column.
*   **Data Preprocessing**:
    *   Missing `Well_ID` values were handled by dropping the column. Missing string values in `Block` and `Village` were filled with 'Unknown'. Missing numerical `Latitude` and `Longitude` values were imputed with their medians. Negative 'F' values were corrected to 0.
    *   Categorical features (`State`, `District`, `Block`, `Village`, `Year`) were one-hot encoded, and numerical features were standardized, resulting in a processed feature set with 16901 dimensions.
    *   Two target variables were prepared: `WQI` for regression and `Water Quality Classification` (5 unique classes) which was label-encoded for classification.
*   **WQI Regression Model Performance**:
    *   A deep learning model with three hidden layers was trained for 50 epochs.
    *   The training loss (MSE) significantly decreased from 49205.0851 in Epoch 1 to 26.8135 in Epoch 50.
    *   The model achieved an outstanding R² score of 0.9994 on the test set.
*   **Water Quality Classification Model Performance**:
    *   A separate deep learning model with three hidden layers was trained for 50 epochs, utilizing class weights to address potential class imbalance (5 classes with counts ranging from 627 to 5238 in the training set).
    *   The training loss (Cross-Entropy) showed a substantial reduction from 0.5772 in Epoch 1 to approximately 0.0000 by Epoch 50.
    *   The model achieved an accuracy of 0.8815 and a weighted F1 score of 0.8837 on the test set.

### Insights or Next Steps
*   The extremely high R² score for the WQI regression model (0.9994) suggests that the model might be overfitting or that the WQI variable is highly correlated with the input features in a way that is easily learnable. Further investigation using techniques like cross-validation or analyzing feature importance could provide more clarity.
*   The Water Quality Classification model shows strong performance with an accuracy and F1 score above 0.88. Given the class imbalance handled during training, analyzing the confusion matrix and per-class metrics would offer deeper insights into the model's performance across different water quality classifications, especially for minority classes.
